In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report
import numpy as np
import seaborn as sns
pd.set_option('display.max_rows', None)

In [5]:
train = pd.read_parquet('data/train.parquet')
test = pd.read_parquet('data/test.parquet')
val = pd.read_parquet('data/val.parquet')

In [6]:
len(val.columns)

264

In [3]:
x_cols = ['Attack ID', 'Avg source IP count', 'Detect count', 'Victim IP', 'Port number', 'Packet speed', 
          'Data speed', 'Avg packet len', 'Source IP count', 
          'Packet speed_normalized', 'Data speed_normalized', 'Avg packet len_normalized', 
          'total_seconds', 'weekday_number', 'time_of_day', 'IsWeekend', 'Start Hour', 'Sin_Hour', 
          'Cos_Hour', 'DayOfYear', 'Sin_DayOfYear', 'Cos_DayOfYear', 'Mean_DataSpeed', 'Std_DataSpeed', 
          'Min_DataSpeed', 'Max_DataSpeed', 'Mean_PacketSpeed', 'Min_PacketSpeed', 
          'Max_PacketSpeed', 'Mean_DetectCount', 'Std_DetectCount', 'Min_DetectCount', 'Max_DetectCount', 
          'VictimIP_Count', 'PortNumber_Count', 'AvgPacketLen_Mean', 'AvgPacketLen_Std', 
          'DataSpeed_PacketSpeed', 'PortFrequency', 'Std_DataSpeed_Replaced', 'Std_DetectCount_Replaced', 
          'AvgPacketLen_Std_Replaced', 'packet_Total', 'PacketSpeed_Per_Second',
          'DataSpeed_Per_TotalSeconds', 'AvgPacketLen_Per_TotalSeconds', 'Is_HTTP', 'Is_HTTPS', 
          'Is_FTP_Control', 'Is_FTP_Data', 'Is_SSH', 'Is_Telnet', 'Is_SMTP', 'Is_DNS', 'Is_POP3',
          'Is_IMAP', 'Is_DHCP', 'Is_SNMP', 'Is_LDAP', 'Is_LDAPS', 'Is_SMB_CIFS', 'Is_RDP', 'Is_SIP', 
          'Is_TFTP', 'Is_MySQL', 'Is_PostgreSQL', 'Is_Oracle', 'Is_HTTP_Alt_8080', 'Is_HTTP_Alt_8081',
          'Is_HTTP_Alt_80', 'Is_HTTPS_Alt_8443', 'Is_Syslog', 'Is_VNC', 'Is_IRC', 'Is_NTP', 'Is_Kerberos', 
          'Is_LDAP_Alt', 'Is_LDAPS_Alt', 'Is_RADIUS', 'Is_PPTP', 'Is_RTSP', 'Is_X11', 'Is_SNMP_Trap', 
          'Is_BGP', 'Is_IMAPS_Alt', 'Is_POP3S_Alt', 'Is_Telnet_SSL', 'Is_NNTP', 'Is_NNTPS', 'Is_LDAP_TLS', 
          'Is_AFS', 'Is_NFS', 'Is_SOCKS', 'Is_RSYNC', 'Is_CUPS', 'Is_TFTP_Alt', 'Is_Modbus', 'Is_CoAP', 
          'Is_MQTT', 'Is_AMQP', 'Is_Redis', 'Is_Memcached', 'Is_Elasticsearch', 'Is_Zookeeper', 
          'Is_Cassandra', 'Is_Docker', 'Is_Kubernetes', 'Is_SMB_Direct', 'Is_iSCSI', 'Is_AFP', 
          'Is_DHCPv6', 'Is_RIPng', 'Is_OSPF', 'Is_PPPoE', 'Is_L2TP', 'Is_GRE', 'Is_ESP', 'Is_AH',
          'PCA_1', 'PCA_2', 'PCA_3', 'PCA_4', 'PCA_5']

In [4]:
# pip install umap-learn
import umap

# 1. Initialize UMAP (n_neighbors controls global vs local structure)
# n_neighbors=15 is default, but 30-50 is often better for features
reducer = umap.UMAP(n_components=2, n_neighbors=30, n_jobs=-1, verbose=True, low_memory=True)

# 2. Learn the mapping from Training data ONLY

train_embedding = reducer.fit_transform(train[x_cols])
                                                        
# 3. Apply the SAME mapping to Test data (Safe for production!)
test_embedding = reducer.transform(test[x_cols])

# 4. Add as features
train[['umap_1', 'umap_2']] = train_embedding
test[['umap_1', 'umap_2']] = test_embedding

UMAP(n_neighbors=30, verbose=True)
Tue May 12 21:06:37 2026 Construct fuzzy simplicial set
Tue May 12 21:06:37 2026 Finding Nearest Neighbors
Tue May 12 21:06:38 2026 Building RP forest with 64 trees
Tue May 12 21:07:19 2026 NN descent for 21 iterations
	 1  /  21
	 2  /  21
	Stopping threshold met -- exiting after 2 iterations
Tue May 12 21:08:51 2026 Finished Nearest Neighbor Search
Tue May 12 21:09:12 2026 Construct embedding


c:\Users\Admin\anaconda3\envs\DS\Lib\site-packages\sklearn\manifold\_spectral_embedding.py:324: UserWarning: Graph is not fully connected, spectral embedding may not work as expected.
  warnings.warn(


KeyboardInterrupt: 

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.manifold import TSNE
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from umap import UMAP

# Assemble into a clean plotting DataFrame
plot_df = pd.DataFrame({
    'UMAP_X': train['umap_1'],
    'UMAP_Y': train['umap_2'],
    'Traffic_Type': train['Type']
})

# Map your numeric classes to clean string labels for the map legend
class_map = {0: 'Suspicious', 1: 'DDoS', 2: 'Normal'}
plot_df['Traffic_Type'] = plot_df['Traffic_Type'].map(class_map)

# --- 3. PAINT THE MAP ---
fig, axes = plt.subplots(1, 2, figsize=(18, 8))
custom_colors = {'Normal': '#2ecc71', 'DDoS': '#e74c3c', 'Suspicious': '#f1c40f'}

# Plot Plot 1: The t-SNE Map
sns.scatterplot(
    data=plot_df, x='TSNE_X', y='TSNE_Y', hue='Traffic_Type',
    palette=custom_colors, alpha=0.4, s=15, ax=axes[0]
)
axes[0].set_title('t-SNE Dimensionality Map', fontsize=14, fontweight='bold')
axes[0].set_xlabel('t-SNE Dimension 1')
axes[0].set_ylabel('t-SNE Dimension 2')

# Plot Plot 2: The UMAP Map
sns.scatterplot(
    data=plot_df, x='UMAP_X', y='UMAP_Y', hue='Traffic_Type',
    palette=custom_colors, alpha=0.4, s=15, ax=axes[1]
)
axes[1].set_title('UMAP Dimensionality Map', fontsize=14, fontweight='bold')
axes[1].set_xlabel('UMAP Dimension 1')
axes[1].set_ylabel('UMAP Dimension 2')

plt.tight_layout()
plt.show()

In [ ]:
train.to_parquet('data/train.parquet', index=False)
test.to_parquet('data/test.parquet', index=False)
##val.to_parquet('data/val.parquet', index=False)